# Data Cleaning & Data Manipulation — PT

## Learning goals

By the end of this lesson, you should be able to:

- identify common data-quality problems;
- clean and standardize column names;
- standardize inconsistent categorical values;
- clean and transform string data;
- convert columns to appropriate data types;
- identify and handle missing values;
- identify and remove duplicate rows;
- verify that a cleaning step produced the expected result.

Throughout the lesson, we will use a simple workflow:

**Inspect → Decide → Clean → Verify**

## 1. Why do we clean data?

Real-world datasets are rarely ready for analysis.

Common problems include:

- inconsistent column names;
- inconsistent category labels;
- numbers stored as strings;
- unwanted characters;
- missing values;
- duplicated rows.

Cleaning data is not just about applying Pandas methods.

Before changing anything, ask:

> **What is wrong with this data, and why does it matter for the analysis?**

We will use an e-commerce dataset for the examples.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "Order ID": [1001, 1002, 1003, 1003, 1004, 1005],
    "Customer City": ["madrid", "Madrid ", "BARCELONA", "BARCELONA", "Valencia", None],
    "Product Category": ["electronics", "Electronics", "home", "home", "Home", "ELECTRONICS"],
    "Price (€)": ["120.50", "89.99", "35.00", "35.00", "49.95", "199.00"],
    "Discount": ["10%", None, "5%", "5%", "0%", "15%"],
    "Units": [1, 2, 3, 3, None, 1]
})

display(orders)

## 2. Inspect before changing

Before cleaning, inspect the DataFrame.

Useful questions include:

- What are the column names?
- What data types does Pandas see?
- Are there missing values?
- Are there duplicates?
- Which categorical values look inconsistent?

In [ ]:
print("Columns:")
print(orders.columns.tolist())

print("\nData types:")
display(orders.dtypes)

print("\nMissing values:")
display(orders.isna().sum())

print("\nDuplicated rows:")
print(orders.duplicated().sum())

### 💡 Quick Check 1 — Complete the code

Use Pandas to inspect:

1. the data types;
2. the number of missing values in each column;
3. the number of duplicated rows.

In [ ]:
# Data types
orders._____

# Missing values per column
orders._____()._____()

# Number of duplicated rows
orders._____()._____()

## 3. Cleaning column names

Column names should be:

- consistent;
- easy to type;
- descriptive;
- free of unnecessary spaces.

A common convention is:

- lowercase;
- spaces replaced with `_`.

For example:

`"Customer City"` → `"customer_city"`

In [ ]:
orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(orders.columns.tolist())

Sometimes a column name also needs a more descriptive replacement.

For individual changes, `rename()` is useful.

In [ ]:
orders = orders.rename(columns={
    "price_(€)": "price_eur"
})

print(orders.columns.tolist())

### 💡 Quick Check 2 — Complete the code

Clean these column names so that they are lowercase and spaces become underscores.

In [ ]:
example_columns = pd.DataFrame({
    "Customer Name": ["Ana"],
    "Order Date": ["2026-08-20"],
    "Total Price": [50]
})

example_columns.columns = (
    example_columns.columns
    .str.________()
    .str.________(" ", "_")
)

print(example_columns.columns.tolist())

## 4. Standardizing inconsistent categorical values

The same category may appear written in different ways:

- `"madrid"`
- `"Madrid "`
- `"MADRID"`

Pandas will treat them as different values.

Before cleaning a categorical column, inspect its unique values.

In [ ]:
print("Customer city values:")
print(orders["customer_city"].unique())

print("\nProduct category values:")
print(orders["product_category"].unique())

### String methods with `.str`

Pandas provides string methods through the `.str` accessor.

Useful methods include:

- `.str.strip()` → remove leading/trailing spaces;
- `.str.lower()` → lowercase;
- `.str.upper()` → uppercase;
- `.str.title()` → title case;
- `.str.replace()` → replace text.

In [ ]:
orders["customer_city"] = (
    orders["customer_city"]
    .str.strip()
    .str.title()
)

orders["product_category"] = (
    orders["product_category"]
    .str.strip()
    .str.title()
)

print(orders["customer_city"].unique())
print(orders["product_category"].unique())

### Mapping categories

Sometimes values are not just formatted differently: they need to be mapped to a common category.

In [ ]:
shipping = pd.Series(["STD", "EXP", "Standard", "Express"])

shipping_map = {
    "STD": "Standard",
    "EXP": "Express",
    "Standard": "Standard",
    "Express": "Express"
}

clean_shipping = shipping.map(shipping_map)

display(clean_shipping)

### 💡 Quick Check 3 — Complete the code

Standardize the following payment methods.

In [ ]:
payments = pd.Series(["CC", "Card", "PAYPAL", "Paypal"])

payment_map = {
    "CC": "Card",
    "Card": "Card",
    "PAYPAL": "Paypal",
    "Paypal": "Paypal"
}

clean_payments = payments.________(payment_map)

display(clean_payments)

## 5. Cleaning string values

Sometimes a string contains characters that should not be part of the final value.

For example, the `discount` column contains percentages as strings.

If we want to calculate with those values, the `%` symbol must be removed first.

In [ ]:
orders["discount_clean"] = (
    orders["discount"]
    .str.replace("%", "", regex=False)
)

display(orders[["discount", "discount_clean"]])

### Splitting strings

`str.split()` can separate pieces of a string using a delimiter.

In [ ]:
tracking_codes = pd.Series(["ES-MAD-001", "FR-PAR-002", "DE-BER-003"])

country_codes = tracking_codes.str.split("-").str[0]

display(country_codes)

### 💡 Quick Check 4 — Complete the code

Extract the middle part of each warehouse code.

In [ ]:
warehouse_codes = pd.Series(["MAD-NORTH-01", "BCN-SOUTH-02", "VAL-EAST-03"])

warehouse_area = warehouse_codes.str.________("-").str[_____]

display(warehouse_area)

## 6. Correcting data types

A column can look numeric while Pandas stores it as text.

Always check the current type before performing calculations.

In [ ]:
display(orders[["price_eur", "discount_clean"]].dtypes)

### `astype()`

If the values are already clean, `astype()` can convert the column.

In [ ]:
orders["price_eur"] = orders["price_eur"].astype(float)

After changing a type, verify the result.

In [ ]:
print(orders["price_eur"].dtype)
print(orders["price_eur"].mean())

### 💡 Quick Check 5 — Complete the code

Convert the following string values to integers.

In [ ]:
units_text = pd.Series(["2", "5", "3", "8"])

units_numeric = units_text.________(_____)

print(units_numeric.dtype)

## 7. Missing values

Missing values represent information that is absent.

In Pandas they often appear as `NaN` or `None`.

The first step is not to delete them automatically.

The first step is to **inspect them**.

In [ ]:
display(orders.isna().sum())

### Choosing a strategy

There is no universal rule for missing values.

Possible strategies include:

- drop rows;
- drop columns;
- fill with a specific value;
- fill a numerical variable with mean or median;
- fill a categorical variable with its mode.

The correct choice depends on what the column means and how it will be used.

### `dropna()`

Use `dropna()` when rows with missing information should not be kept.

For example, suppose a missing city makes an order unusable for a geographical analysis:

In [ ]:
orders_without_missing_city = orders.dropna(subset=["customer_city"])

display(orders_without_missing_city)

### `fillna()`

Sometimes we prefer to replace missing values.

For example, a missing discount can reasonably mean that no discount was applied:

In [ ]:
orders["discount_clean"] = orders["discount_clean"].fillna("0")
orders["discount_clean"] = orders["discount_clean"].astype(float)

display(orders[["discount", "discount_clean"]])

For numerical variables, mean or median may **sometimes** be appropriate.

For categorical variables, the mode is **sometimes** appropriate.

These choices should be justified rather than applied automatically.

In [ ]:
median_units = orders["units"].median()
orders["units_filled"] = orders["units"].fillna(median_units)

display(orders[["units", "units_filled"]])

### Verify after cleaning

Always check that the operation had the intended effect.

In [ ]:
display(orders[["discount_clean", "units_filled"]].isna().sum())

### 💡 Quick Check 6 — Complete the code

Fill the missing values in `stock` with the median, then verify that no missing values remain.

In [ ]:
inventory = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "stock": [10, None, 6, 14]
})

median_stock = inventory["stock"].________()

inventory["stock"] = inventory["stock"].________(median_stock)

inventory["stock"].________().________()

## 8. Duplicates

Duplicated rows can accidentally count the same observation more than once.

Use `duplicated()` to identify them.

In [ ]:
duplicate_mask = orders.duplicated()

display(duplicate_mask)
print("Number of duplicated rows:", duplicate_mask.sum())

### Removing duplicates

`drop_duplicates()` removes duplicated rows.

By default, Pandas keeps the first occurrence.

In [ ]:
orders_no_duplicates = orders.drop_duplicates()

print("Rows before:", len(orders))
print("Rows after:", len(orders_no_duplicates))

Duplicates can also be evaluated using specific columns with the `subset` parameter.

For example, in a real dataset you might decide that an order ID should be unique:

In [ ]:
orders_by_id = orders.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

display(orders_by_id)

After dropping rows, the index may contain gaps.

`reset_index(drop=True)` creates a new sequential index.

In [ ]:
orders_by_id = orders_by_id.reset_index(drop=True)

display(orders_by_id)

### 💡 Quick Check 7 — Complete the code

Remove duplicated rows from the DataFrame and reset its index.

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 2, 3],
    "city": ["Madrid", "Paris", "Paris", "Berlin"]
})

customers = customers.________()
customers = customers.________(drop=True)

display(customers)

## 9. Dropping unnecessary columns

Sometimes a column is not needed for the current analysis.

Use `drop(columns=...)` to remove it.

Do not remove columns simply because they look inconvenient. First decide whether they are relevant to the analysis.

In [ ]:
orders_reduced = orders.drop(columns=["discount"])

display(orders_reduced.head())

### 💡 Quick Check 8 — Complete the code

Remove the `temporary_note` column.

In [ ]:
products = pd.DataFrame({
    "product": ["A", "B", "C"],
    "price": [10, 20, 30],
    "temporary_note": ["check", "check", "check"]
})

products = products.________(columns=[________])

display(products)

## 10. The cleaning workflow

A reliable cleaning process follows a cycle:

### 1. Inspect

Identify the problem.

### 2. Decide

Ask what the correct representation should be.

### 3. Clean

Apply the appropriate transformation.

### 4. Verify

Check that the result is what you expected.

Cleaning without verification can silently introduce new problems.

## 11. Final Challenge — Complete the cleaning steps

A small product catalog has several data-quality problems.

Complete the missing code.

In [ ]:
catalog = pd.DataFrame({
    "Product Name": ["Keyboard", "Mouse", "Mouse", "Monitor"],
    "Category": ["electronics", "Electronics ", "Electronics ", "OFFICE"],
    "Price": ["49.99", "25.50", "25.50", "189.00"],
    "Stock": [10, None, None, 4]
})

# 1. Clean column names
catalog.columns = (
    catalog.columns
    .str.________()
    .str.________(" ", "_")
)

# 2. Standardize categories
catalog["category"] = (
    catalog["category"]
    .str.________()
    .str.________()
)

# 3. Convert price to numeric
catalog["price"] = catalog["price"].________(_____)

# 4. Fill missing stock values with the median
median_stock = catalog["stock"].________()
catalog["stock"] = catalog["stock"].________(median_stock)

# 5. Remove duplicated rows
catalog = catalog.________()

# 6. Reset the index
catalog = catalog.________(drop=True)

display(catalog)

## Summary

Today you learned how to approach common data-quality problems using a simple workflow:

**Inspect → Decide → Clean → Verify**

You practiced:

- cleaning column names;
- standardizing categorical values;
- manipulating strings;
- converting data types;
- handling missing values;
- removing duplicates;
- dropping unnecessary columns.

### Further exploration

Pandas provides additional options for advanced cleaning, including:

- `dropna()` with `how` and `thresh`;
- forward fill and backward fill;
- more advanced `apply()` transformations;
- regular expressions;
- date handling;
- advanced imputation strategies.

These tools can be introduced when a dataset requires them.